# Heart Disease Risk Prediction - Exploratory Data Analysis & Model Training

**Educational Project for Campus Placements / College Resume**  
*Author: Surya | Technologies: Python, Pandas, NumPy, Scikit-learn, Matplotlib, Seaborn*

---

> **Disclaimer:** This notebook is an educational demonstration of machine-learning classification and data analysis. It is **not a medical diagnostic tool**.

## 1. Import Libraries
We begin by importing the core data science stack: Pandas for tabular manipulation, NumPy for array mathematics, Scikit-learn for machine learning modeling, and Matplotlib/Seaborn for visual data exploration.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

# Plot styling
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
print('Libraries loaded successfully!')

## 2. Load the Dataset
We load the benchmark **UCI Cleveland Heart Disease dataset** located in `../data/heart_disease.csv`.

In [ ]:
# Relative path to dataset
data_path = Path('../data/heart_disease.csv')
df = pd.read_csv(data_path)

print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

## 3. Data Integrity & Missing Values Check
Before modeling, we inspect data types, summary statistics, and check for any NULL or missing values.

In [ ]:
print('Missing Values per Column:')
print(df.isnull().sum())
print('\nSummary Statistics:')
df.describe().T[['min', 'mean', 'max']]

## 4. Exploratory Data Analysis (EDA)
Let us explore the distribution of the target variable (`0 = No Disease/Low Risk`, `1 = Heart Disease Likelihood`) and feature correlations.

In [ ]:
# Target distribution
plt.figure(figsize=(6, 4))
ax = sns.countplot(x='target', data=df, hue='target', palette=['#10b981', '#ef4444'], legend=False)
plt.title('Target Class Distribution')
plt.xlabel('Heart Disease Status (0 = Lower Risk, 1 = Higher Risk)')
plt.ylabel('Patient Count')
plt.show()

In [ ]:
# Feature Correlation Heatmap
plt.figure(figsize=(11, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', annot=True, fmt='.2f', square=True, linewidths=0.5)
plt.title('Correlation Matrix Heatmap')
plt.show()

## 5. Preprocessing & Data Leakage Prevention
A critical interview concept: **Never fit the scaler on the entire dataset.**
1. Split the dataset into features ($X$) and target ($y$).
2. Perform stratified split (80% training, 20% testing).
3. Fit `StandardScaler` **only** on $X_{\text{train}}$, then transform both $X_{\text{train}}$ and $X_{\text{test}}$.

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

# Stratified split ensures equal class representation
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Testing set:  {X_test.shape[0]} samples')

# Fit scaler ONLY on training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Features scaled successfully with zero data leakage!')

## 6. Model Training & Comparison
We train:
1. **Logistic Regression** (Primary Model) - Interpretable linear classifier that computes decision probabilities.
2. **Random Forest Classifier** (Comparison Model) - Ensemble of decision trees.

In [ ]:
# 1. Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

# 2. Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print('Both models trained successfully!')

## 7. Model Evaluation
We evaluate performance using Accuracy, Precision, Recall, F1-Score, and Confusion Matrix.

In [ ]:
results = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Logistic Regression': [
        f'{accuracy_score(y_test, y_pred_lr)*100:.2f}%',
        f'{precision_score(y_test, y_pred_lr)*100:.2f}%',
        f'{recall_score(y_test, y_pred_lr)*100:.2f}%',
        f'{f1_score(y_test, y_pred_lr)*100:.2f}%'
    ],
    'Random Forest': [
        f'{accuracy_score(y_test, y_pred_rf)*100:.2f}%',
        f'{precision_score(y_test, y_pred_rf)*100:.2f}%',
        f'{recall_score(y_test, y_pred_rf)*100:.2f}%',
        f'{f1_score(y_test, y_pred_rf)*100:.2f}%'
    ]
})
results

In [ ]:
# Plot Confusion Matrix for Logistic Regression
cm = confusion_matrix(y_test, y_pred_lr)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted: 0', 'Predicted: 1'],
            yticklabels=['Actual: 0', 'Actual: 1'])
plt.title('Confusion Matrix - Logistic Regression')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

## 8. Saving Artifacts for Flask Deployment
We save the trained Logistic Regression model and fitted `StandardScaler` into a serialized bundle using `joblib`.

In [ ]:
bundle = {
    'model': lr,
    'comparison_model': rf,
    'scaler': scaler,
    'feature_names': list(X.columns)
}
Path('../model').mkdir(parents=True, exist_ok=True)
joblib.dump(bundle, '../model/heart_disease_model.pkl')
print('Model bundle exported to ../model/heart_disease_model.pkl')

## 9. Conclusion & Key Takeaways
- **High Recall (90.91%)**: The Logistic Regression model demonstrates high sensitivity, minimizing false negatives (missing patients with cardiac risk).
- **Interpretability**: Logistic Regression coefficients provide clear insights into clinical risk directions.
- **Pipeline Parity**: Using `StandardScaler` both during training and web inference guarantees zero distribution shift.